In [ ]:
SAMPLE_RESUME = """
张三
zhangsan@163.com | +86 138-0000-1234 | 北京, 中国
linkedin.com/in/zhangsan | github.com/zhangsan

个人简介
资深 Python 开发工程师，拥有 7 年构建可扩展 Web 应用和数据管道
的经验。曾在 B 轮创业公司带领 5-8 人的工程团队。

工作经历
高级软件工程师 | 北京科技有限公司 | 2021-至今
- 使用 FastAPI + Kubernetes 设计并实现了支撑日均 1000 万次请求的微服务架构平台
- 通过 Redis 缓存和异步优化，将 API 延迟降低 40%
- 主导了从单体架构向微服务的迁移（12 个月项目，5 名工程师）

软件工程师 | 数据流系统公司 | 2018-2021
- 使用 Apache Spark 和 Airflow 构建处理 500GB/天数据的 ML 数据管道
- 基于 Django REST Framework 开发 REST API，服务每日 5 万用户

技能
编程语言：Python, JavaScript, SQL, Bash
框架：FastAPI, Django, React, Spark
工具：Docker, Kubernetes, Redis, PostgreSQL, Git, Airflow
云服务：AWS (EC2, S3, RDS, Lambda)

教育经历
计算机科学学士 | 北京大学 | 2017

CERTIFICATIONS
AWS 解决方案架构师助理认证
"""

In [ ]:
PARSE_PROMPT = """从这份简历中提取结构化信息，并返回 JSON：
{
    "name": "姓名",
    "email": "邮箱或 null",
    "phone": "电话或 null",
    "location": "城市, 国家或 null",
    "linkedin": "URL 或 null",
    "github": "URL 或 null",
    "summary": "2-3 句个人简介",
    "years_experience": 数字,
    "current_title": "当前/最新职位",
    "skills": {
        "languages": ["Python", "JavaScript", ...],
        "frameworks": ["Django", "React", ...],
        "tools": ["Docker", "Git", ...],
        "soft_skills": ["领导力", ...]
    },
    "experience": [{"title": "...", "company": "...", "duration": "...", "highlights": ["..."]}],
    "education": [{"degree": "...", "institution": "...", "year": "..."}],
    "certifications": ["..."],
    "languages_spoken": ["中文", ...]
}
仅返回合法的 JSON，不要包含其他内容。"""

FIT_PROMPT = """根据候选人画像和职位描述，返回 JSON：
{
    "fit_score": 0-100,
    "fit_label": "Excellent|Good|Fair|Poor",
    "strengths": ["匹配点 1", "匹配点 2", ...],
    "gaps": ["缺失技能 1", ...],
    "recommendation": "Hire|Consider|Pass",
    "recommendation_reason": "2-3 句理由说明"
}
仅返回合法的 JSON，不要包含其他内容。"""

In [3]:
import re
import json

def parse_json_response(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)
    return json.loads(cleaned)

In [4]:
def read_resume_text(path: str) -> str:
    if path.endswith(".pdf"):
        try:
            import pypdf
            with open(path, "rb") as f:
                reader = pypdf.PdfReader(f)
                return "\n".join(page.extract_text() for page in reader.pages)
        except ImportError:
            print("⚠️  pypdf not installed. Install with: pip install pypdf")
            raise
    with open(path) as f:
        return f.read()

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

from dotenv import load_dotenv
import os

load_dotenv()

model = os.getenv('MODEL_NAME', 'gpt-5.5')

llm = ChatOpenAI(model=model, temperature=0, extra_body={'enable_thinking': False})


def parse_resume(text: str) -> dict:
    messages = [SystemMessage(content=PARSE_PROMPT), HumanMessage(content=f"简历：\n{text}")]
    response = llm.invoke(messages)
    return parse_json_response(response.content)


def score_fit(profile: dict, job_desc: str) -> dict:
    messages = [
        SystemMessage(content=FIT_PROMPT),
        HumanMessage(content=f"候选人画像: \n{json.dumps(profile, indent=2)}\n\nJob description:\n{job_desc}"),
    ]
    response = llm.invoke(messages)
    return parse_json_response(response.content)

In [8]:
profile = parse_resume(text=SAMPLE_RESUME)

print("简历：")
print(json.dumps(profile, ensure_ascii=False, indent=2))

result = score_fit(profile=profile, job_desc="前端开发工程师，要求熟悉 JavaScript")

print("\n匹配度：")
print(json.dumps(result, ensure_ascii=False, indent=2))

简历：
{
  "name": "张三",
  "email": "zhangsan@163.com",
  "phone": "+86 138-0000-1234",
  "location": "北京, 中国",
  "linkedin": "linkedin.com/in/zhangsan",
  "github": "github.com/zhangsan",
  "summary": "资深 Python 开发工程师，拥有 7 年构建可扩展 Web 应用和数据管道的经验。曾在 B 轮创业公司带领 5-8 人的工程团队。",
  "years_experience": 7,
  "current_title": "高级软件工程师",
  "skills": {
    "languages": [
      "Python",
      "JavaScript",
      "SQL",
      "Bash"
    ],
    "frameworks": [
      "FastAPI",
      "Django",
      "React",
      "Spark"
    ],
    "tools": [
      "Docker",
      "Kubernetes",
      "Redis",
      "PostgreSQL",
      "Git",
      "Airflow"
    ],
    "soft_skills": []
  },
  "experience": [
    {
      "title": "高级软件工程师",
      "company": "北京科技有限公司",
      "duration": "2021-至今",
      "highlights": [
        "使用 FastAPI + Kubernetes 设计并实现了支撑日均 1000 万次请求的微服务架构平台",
        "通过 Redis 缓存和异步优化，将 API 延迟降低 40%",
        "主导了从单体架构向微服务的迁移（12 个月项目，5 名工程师）"
      ]
    },
    {
      "title": "软件工程师",
      "comp